# 05 · Gerçek T-H-E ile 29 harflik Türkçe Keras CNN

Bu defter **yalnız gerçek insan el yazısı** olan `version6.csv` dosyasını kullanır. Artırılmış sürümler, font-render veri ve karakterlerden oluşturulmuş sahte satırlar kullanılmaz. Kaynak makalede `public-domain/free-to-use` ifadeleri bulunsa da depo açık bir SPDX lisansı yayımlamıyor; bu nedenle veri/model yeniden dağıtılmamalı, yalnız izinli/özel kullanım varsayılmalıdır.

Yayımlanan dosyada büyük `Ç` ve `Ş` satırları yoktur. Büyük/küçük el yazısı zaten görsel olarak belirsiz olduğundan model 29 harflik casefold sınıfları öğrenir; büyük harf bağlamdan geri kurulur. T-H-E yazar kimliği vermediği için test bölümü yazar-ayrık değildir.

In [ ]:
!apt-get -qq update && apt-get -qq install -y git-lfs
!pip install -q "tensorflow>=2.16,<2.22" scikit-learn pandas seaborn opencv-python-headless
!rm -rf /content/thedataset
!git clone -q https://github.com/bartosgaye/thedataset /content/thedataset
!cd /content/thedataset && git lfs install && git lfs pull --include="version6.csv"
!test $(wc -c < /content/thedataset/version6.csv) -gt 1000000

In [ ]:
import json, os, random, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
CSV = Path('/content/thedataset/version6.csv')
THE = {1:'a',2:'b',3:'c',4:'d',5:'e',6:'f',7:'g',8:'h',9:'i',10:'j',11:'k',12:'l',13:'m',14:'n',15:'o',16:'p',17:'q',18:'r',19:'s',20:'t',21:'u',22:'v',23:'w',24:'x',25:'y',26:'z',27:'ç',28:'ğ',29:'ı',30:'ş',31:'ö',32:'ü',40:'A',41:'B',42:'C',43:'D',44:'E',45:'F',46:'G',47:'H',48:'I',49:'J',50:'K',51:'L',52:'M',53:'N',54:'O',55:'P',56:'Q',57:'R',58:'S',59:'T',60:'U',61:'V',62:'W',63:'X',64:'Y',65:'Z',66:'Ç',67:'Ğ',68:'İ',69:'Ş',70:'Ö',71:'Ü'}
TR_LOWER = 'abcçdefgğhıijklmnoöprsştuüvyz'
TR_UPPER = 'ABCÇDEFGĞHIİJKLMNOÖPRSŞTUÜVYZ'
TR = set(TR_LOWER + TR_UPPER)
etiket_ids = [k for k,v in THE.items() if v in TR]
assert len(etiket_ids) == 58
print('TensorFlow', tf.__version__, 'GPU', tf.config.list_physical_devices('GPU'))

In [ ]:
df = pd.read_csv(CSV, header=None, dtype=np.uint8)
df = df[df[0].isin(etiket_ids)].reset_index(drop=True)
sayim=df[0].value_counts().to_dict(); assert set(sayim.values()) == {1000}, sayim
mevcut_ids=[k for k in etiket_ids if k in sayim]
eksik=[THE[k] for k in etiket_ids if k not in sayim]
sinif_sirasi=[THE[k] for k in mevcut_ids]; id_to_index={k:i for i,k in enumerate(mevcut_ids)}
print('Fiilî Türkçe sınıf:',len(sinif_sirasi),'örnek:',len(df),'eksik:',eksik)
assert len(df)==56_000 and set(eksik)=={'Ç','Ş'}, 'Kaynak dosya değişti; envanteri yeniden doğrulayın'
# CSV, görüntüyü sütun-major düzleştirdiği için 28×28 eksenlerini düzelt.
x = df.iloc[:,1:785].to_numpy(dtype=np.float32).reshape(-1,28,28).transpose(0,2,1)
if x.max() > 1: x /= 255.0
# Uygulamadaki _karakter_girdisi ile aynı: glifi sıkı kırp, oranı koru,
# en uzun kenarı 24 px yap ve 28×28 tuvalde ortala.
def karakter_girdisi(ink):
    ys,xs=np.where(ink>0)
    if len(xs)==0: return np.zeros((28,28,1),dtype=np.float32)
    kirp=ink[ys.min():ys.max()+1,xs.min():xs.max()+1]
    olcek=min(24/max(kirp.shape[1],1),24/max(kirp.shape[0],1))
    yeni=(max(1,round(kirp.shape[1]*olcek)),max(1,round(kirp.shape[0]*olcek)))
    kirp=cv2.resize(kirp,yeni,interpolation=cv2.INTER_AREA)
    tuval=np.zeros((28,28),dtype=np.float32)
    ox=(28-kirp.shape[1])//2; oy=(28-kirp.shape[0])//2
    tuval[oy:oy+kirp.shape[0],ox:ox+kirp.shape[1]]=kirp
    return tuval[...,None]
x=np.stack([karakter_girdisi(im) for im in x]).astype(np.float32)
y = df[0].map(id_to_index).to_numpy(dtype=np.int32)
del df
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.10, random_state=SEED, stratify=y)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=.10, random_state=SEED, stratify=y_train)
base_order=list(TR_LOWER); upper_to_lower=dict(zip(TR_UPPER,TR_LOWER))
class_to_base=np.array([base_order.index(upper_to_lower.get(ch,ch)) for ch in sinif_sirasi],dtype=np.int32)
y_train,y_val,y_test=class_to_base[y_train],class_to_base[y_val],class_to_base[y_test]
print(x_train.shape, x_val.shape, x_test.shape)
print('UYARI: split sample-random; writer kimliği olmadığı için writer-disjoint değil.')

In [ ]:
from tensorflow import keras
L = keras.layers
model = keras.Sequential([
    L.Input((28,28,1)),
    L.Conv2D(64,3,padding='same',activation='relu'),
    L.Conv2D(64,3,padding='same',activation='relu'), L.MaxPool2D(), L.Dropout(.12),
    L.Conv2D(128,3,padding='same',activation='relu'),
    L.Conv2D(128,3,padding='same',activation='relu'), L.MaxPool2D(), L.Dropout(.18),
    L.Conv2D(256,3,padding='same',activation='relu'),
    L.Flatten(), L.Dense(512,activation='relu'), L.Dropout(.30),
    L.Dense(len(base_order), activation='softmax')
])
model.compile(optimizer=keras.optimizers.Adam(8e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()
callbacks = [
    keras.callbacks.ModelCheckpoint('/content/en_iyi.keras', monitor='val_accuracy', save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=.35, patience=2, min_lr=1e-6),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
]
history = model.fit(x_train,y_train,validation_data=(x_val,y_val),epochs=35,batch_size=256,callbacks=callbacks)

In [ ]:
model = keras.models.load_model('/content/en_iyi.keras')
loss, acc = model.evaluate(x_test, y_test, verbose=0)
probs = model.predict(x_test, batch_size=512, verbose=0)
pred=probs.argmax(1); conf=probs.max(1); correct=(pred==y_test)
print(f'T-H-E casefold sample-random test accuracy: {acc:.4%}')
rapor = classification_report(y_test, pred, target_names=base_order, output_dict=True, zero_division=0)
order=np.argsort(-conf); cum_acc=np.cumsum(correct[order])/np.arange(1,len(correct)+1)
selective={}
for target in [0.99,0.995,0.999]:
    valid=np.where(cum_acc>=target)[0]
    if len(valid):
        k=int(valid[-1]+1); threshold=float(conf[order][k-1])
        selective[str(target)]={'threshold':threshold,'coverage':k/len(correct),'accepted':k,'accuracy':float(cum_acc[k-1])}
        print(target,selective[str(target)])
sonuc = {'test_accuracy':float(acc),'test_loss':float(loss),'writer_disjoint':False,'classes':len(base_order),'samples':len(x),'casefolded':True,'missing_source_classes':eksik,'uppercase_recovery':'context-required','augmentation_used':False,'selective':selective,'classification_report':rapor}
Path('/content/etiketler.json').write_text(json.dumps(base_order,ensure_ascii=False,indent=2),encoding='utf-8')
Path('/content/karakter_metrikleri.json').write_text(json.dumps(sonuc,ensure_ascii=False,indent=2),encoding='utf-8')
model.save('/content/turkce_karakter_cnn.keras')

In [ ]:
import shutil
from google.colab import drive, files
OUT = Path('/content/karakter_cnn')
OUT.mkdir(parents=True, exist_ok=True)
for ad in ['turkce_karakter_cnn.keras','etiketler.json','karakter_metrikleri.json']:
    shutil.copy2(Path('/content',ad), OUT/ad)
arsiv = shutil.make_archive('/content/turkce_karakter_cnn','zip',root_dir=OUT)
try:
    drive.mount('/content/drive', force_remount=True)
    DRIVE_OUT = Path('/content/drive/MyDrive/sinav_okur/modeller/karakter_cnn_entegre')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    for ad in ['turkce_karakter_cnn.keras','etiketler.json','karakter_metrikleri.json','turkce_karakter_cnn.zip']:
        shutil.copy2(Path('/content',ad), DRIVE_OUT/ad)
    print('Drive kaydı:', DRIVE_OUT)
except Exception as exc:
    print('Drive bağlanamadı; tarayıcı indirmesi kullanılacak:', repr(exc))
files.download(arsiv)
print('Bu model yalnız ayrık karakter benchmarkıdır; bağlı sınav satırları için 06 defterini kullanın.')